In [0]:
import requests
import os
from pyspark.sql import functions as F

BASE_URL    = "https://d37ci6vzurychx.cloudfront.net/trip-data"
VOLUME_PATH = "/Volumes/nyc_taxi_datalake/bronze/raw_files/yellow_taxi"
DELTA_TABLE = "nyc_taxi_datalake.bronze.yellow_taxi"
ZONE_TABLE  = "nyc_taxi_datalake.bronze.taxi_zones"

# full year
YEARS_MONTHS = [(2024, m) for m in range(1, 13)]

In [0]:
def already_ingested(year: int, month: int) -> bool:
    try:
        count = spark.table(DELTA_TABLE) \
                     .filter(
                         (F.col("_pickup_year")  == year) &
                         (F.col("_pickup_month") == month)
                     ).limit(1).count()
        return count > 0
    except Exception:
        return False

## Downloading Function


In [0]:
def download_to_volume(year: int, month: int) -> str:
    filename    = f"yellow_tripdata_{year}-{month:02d}.parquet"
    url         = f"{BASE_URL}/{filename}"
    volume_dir  = f"{VOLUME_PATH}/{year}/{month:02d}"
    volume_file = f"{volume_dir}/{filename}"

    if os.path.exists(volume_file):
        print(f" Already exists, skipping: {filename}")
        return volume_file

    os.makedirs(volume_dir, exist_ok=True)

    print(f" Downloading {filename}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(volume_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"Saved to: {volume_file}")
    return volume_file

## **Ingest Function**

In [0]:
def ingest_to_bronze(year: int, month: int):
    filename    = f"yellow_tripdata_{year}-{month:02d}.parquet"
    volume_file = f"{VOLUME_PATH}/{year}/{month:02d}/{filename}"

    df = spark.read.parquet(volume_file)

    df = df \
        .withColumn("_ingested_at",  F.current_timestamp()) \
        .withColumn("_source_file",  F.lit(filename)) \
        .withColumn("_pickup_year",  F.lit(year)) \
        .withColumn("_pickup_month", F.lit(month))

    row_count = df.count()

    df.write \
        .format("delta") \
        .mode("append") \
        .partitionBy("_pickup_year", "_pickup_month") \
        .option("mergeSchema", "true") \
        .saveAsTable(DELTA_TABLE)

    print(f"{year}-{month:02d} → {row_count:,} rows written to Bronze")

In [0]:
for year, month in YEARS_MONTHS:
    print(f"\n{'='*40}")
    print(f"Processing {year}-{month:02d}")
    print(f"{'='*40}")

    if already_ingested(year, month):
        print(f"⏭  Already ingested, skipping {year}-{month:02d}")
        continue

    download_to_volume(year, month)
    ingest_to_bronze(year, month)

print("\n Bronze ingestion complete!")

In [0]:
df_check = spark.table(DELTA_TABLE)

print(f"Total rows in Bronze: {df_check.count():,}")

print("\nPartitions loaded:")
df_check.select("_pickup_year", "_pickup_month") \
        .distinct() \
        .orderBy("_pickup_year", "_pickup_month") \
        .show()

print("\nSample rows:")
df_check.show(3, truncate=False)

print("\nSchema:")
df_check.printSchema()

## Zone Lookup


In [0]:
zone_url    = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
volume_dir  = "/Volumes/nyc_taxi_datalake/bronze/raw_files"
volume_file = f"{volume_dir}/taxi_zone_lookup.csv"

os.makedirs(volume_dir, exist_ok=True)

print("⬇️  Downloading taxi_zone_lookup.csv...")
response = requests.get(zone_url)
response.raise_for_status()

with open(volume_file, "wb") as f:
    f.write(response.content)

print(" Saved to Volume")

zone_df = spark.read \
               .option("header", "true") \
               .option("inferSchema", "true") \
               .csv(volume_file)

zone_df.write \
       .format("delta") \
       .mode("overwrite") \
       .saveAsTable(ZONE_TABLE)

print("Zone lookup table saved")
zone_df.show(5)